# Context budgeting for RAG

[Português (Brasil)](orcamento_contexto.ipynb) · [Español](orcamento_contexto.es.ipynb)

Select passages by relevance, cap each source, and check the size of the JSON that will be sent.

[Full article](../article/ARTICLE.en.md)

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

cwd = Path.cwd()
lab = next(
    path
    for path in (cwd, cwd.parent, cwd / "0013_orcamento_contexto")
    if path.name == "0013_orcamento_contexto" and (path / "src" / "main.py").is_file()
)

def run(*options):
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", str(lab / "src" / "main.py"), *options],
        capture_output=True, check=True, text=True, encoding="utf-8",
    )
    return json.loads(completed.stdout)

Run the next cell to reproduce the experiment and check the results with assertions.

At 245 bytes, the encoded `context` field takes 214 bytes and contains two passages. Check `skipped` to distinguish the source quota from the byte limit. At 120 bytes, only `refund-summary` fits.

In [ ]:
report = run("--bytes", "245", "--per-source", "1")
assert len(report["context"].encode("utf-8")) == report["bytes_used"] == 214
assert [item["id"] for item in json.loads(report["context"])["documents"]] == [
    "refund-summary", "refund-window"
]
assert {item["reason"] for item in report["skipped"]} == {"source_limit", "byte_budget"}
smaller = run("--bytes", "120")
assert [item["id"] for item in json.loads(smaller["context"])["documents"]] == ["refund-summary"]
report